# Step 7 — Conditional Volatility of Gilts (GARCH / GJR-GARCH)

Tests the article's *second* quantitative claim — that gilts have become "unnervingly volatile" — by modelling the conditional variance of daily 10-year yield changes for the UK, Germany and the US, and asking whether the UK shows an asymmetric, sell-off-driven volatility response that its peers do not.

**Script notes**

```
Step 7 - Conditional volatility (time-varying volatility pillar)
The PCA work (steps 2-6) addresses CO-MOVEMENT and LEVELS but is silent on
VOLATILITY. The article also claims gilts are 'unnervingly volatile' and that
'political headlines feed directly into funding costs'. That is a statement
about (i) the level and persistence of conditional volatility and (ii) its
ASYMMETRY - whether yield rises (sell-offs, bad news for bondholders) raise
volatility more than yield falls.

Method:
  - series: daily CHANGES in bp of UK_10y, DE_10y, US_10y (from changes_bp.csv)
  - GARCH(1,1)  : symmetric benchmark, Student-t innovations
  - GJR-GARCH(1,1): adds gamma * I(eps>0) * eps^2  (asymmetry term)
        NB yields are modelled directly, so a POSITIVE shock = yield RISE = sell-off.
        gamma<0 here => sell-offs raise next-day variance MORE than rallies.
  - model choice by AIC (and gamma t-stat); persistence and shock half-life
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import sys
!{sys.executable} -m pip install arch


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from arch import arch_model

In [3]:
ch = pd.read_csv('./changes_bp.csv', index_col=0, parse_dates=True)
SERIES = ['UK_10y', 'DE_10y', 'US_10y']
X = ch[SERIES].dropna()
print(f'sample: {X.index.min().date()} -> {X.index.max().date()}   n={len(X)}')

sample: 2024-01-05 -> 2026-05-29   n=519


## 6.1 &nbsp;Fit symmetric GARCH(1,1) and asymmetric GJR-GARCH(1,1)
Student-t innovations throughout (yield changes are fat-tailed). The mean is a constant; we are modelling the conditional **variance**, not predicting the level.

In [4]:
def fit_pair(y):
    g  = arch_model(y, mean='Constant', vol='GARCH', p=1, q=1, dist='t').fit(disp='off')
    gj = arch_model(y, mean='Constant', vol='GARCH', p=1, o=1, q=1, dist='t').fit(disp='off')
    return g, gj
fits = {s: fit_pair(X[s]) for s in SERIES}
print('Fitted GARCH(1,1) and GJR-GARCH(1,1) for', ', '.join(SERIES))

Fitted GARCH(1,1) and GJR-GARCH(1,1) for UK_10y, DE_10y, US_10y


## 6.2 &nbsp;Model comparison and parameters
For GJR the persistence is `alpha + 0.5*gamma + beta`; the shock half-life is `ln(0.5)/ln(persistence)` trading days.

In [5]:
rows = []
for s in SERIES:
    g, gj = fits[s]
    ag, aj = g.params, gj.params
    pers_g  = ag.get('alpha[1]',0) + ag.get('beta[1]',0)
    pers_gj = aj.get('alpha[1]',0) + 0.5*aj.get('gamma[1]',0) + aj.get('beta[1]',0)
    hl = np.log(0.5)/np.log(pers_gj) if 0 < pers_gj < 1 else np.nan
    gamma = aj.get('gamma[1]', np.nan)
    t_gamma = gamma / gj.std_err['gamma[1]']
    rows.append({
        'series': s,
        'GARCH_AIC': round(g.aic,1),
        'GJR_AIC': round(gj.aic,1),
        'AIC_favours': 'GJR' if gj.aic < g.aic else 'GARCH',
        'gamma': round(gamma,4),
        't(gamma)': round(t_gamma,2),
        'persist_GJR': round(pers_gj,3),
        'half_life_d': round(hl,1),
    })
comp = pd.DataFrame(rows).set_index('series')
print(comp.to_string())

        GARCH_AIC  GJR_AIC AIC_favours   gamma  t(gamma)  persist_GJR  half_life_d
series                                                                            
UK_10y     3234.8   3230.3         GJR -0.0829     -1.81        0.961         17.3
DE_10y     3031.8   3033.1       GARCH -0.0367     -0.93        0.982         38.8
US_10y     3260.3   3261.9       GARCH  0.0504      0.47        0.736          2.3


**Reading the table.** A negative `gamma` for the UK means an upward yield move — a gilt sell-off, the bad-news direction for bondholders — raises the next day's conditional variance by *more* than a rally of the same size. The asymmetry term improves AIC for the UK but not for Germany or the US, so this leverage-type response is a UK-specific feature rather than a common one. The evidence is directionally clear but statistically modest (see the t-statistic), and is reported as such.

In [6]:
# UK focus: print the full GJR summary for the appendix
print(fits['UK_10y'][1].summary())

                      Constant Mean - GJR-GARCH Model Results                       
Dep. Variable:                       UK_10y   R-squared:                       0.000
Mean Model:                   Constant Mean   Adj. R-squared:                  0.000
Vol Model:                        GJR-GARCH   Log-Likelihood:               -1609.17
Distribution:      Standardized Student's t   AIC:                           3230.33
Method:                  Maximum Likelihood   BIC:                           3255.84
                                              No. Observations:                  519
Date:                      Thu, Jun 11 2026   Df Residuals:                      518
Time:                              08:39:14   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             0

## 6.3 &nbsp;Conditional volatility paths
Annualised conditional volatility (bp/day x sqrt(252)) from the preferred model for each series — GJR for the UK, GARCH for the peers.

In [7]:
def cond_vol(res):
    return res.conditional_volatility * np.sqrt(252)
cv = pd.DataFrame({
    'UK': cond_vol(fits['UK_10y'][1]),   # GJR
    'DE': cond_vol(fits['DE_10y'][0]),   # GARCH
    'US': cond_vol(fits['US_10y'][0]),   # GARCH
}, index=X.index)
print('Annualised conditional vol (bp), summary:')
print(cv.describe().loc[['mean','min','max']].round(0).to_string())
print('\nMay 2026 mean conditional vol (bp):')
print(cv.loc['2026-05-01':'2026-05-29'].mean().round(0).to_string())

Annualised conditional vol (bp), summary:
         UK     DE     US
mean   87.0   72.0   89.0
min    63.0   52.0   77.0
max   126.0  117.0  128.0

May 2026 mean conditional vol (bp):
UK    101.0
DE     68.0
US     87.0


In [8]:
cv.to_csv('./garch_condvol.csv')
comp.to_csv('./garch_model_comparison.csv')
print('\nsaved garch_condvol.csv, garch_model_comparison.csv')


saved garch_condvol.csv, garch_model_comparison.csv
